# DSA 210 – Internet Usage and Economic Development
## EDA & Hypothesis Testing
**Student:** Serkan Evleksiz  
**Dataset:** OECD Countries, 2005–2020  
**Sources:** World Bank Open Data, Our World in Data, OECD

---
## 0. Setup & Data Loading

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings
import os

warnings.filterwarnings('ignore')
os.makedirs('../figures', exist_ok=True)

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams.update({'figure.dpi': 120, 'font.size': 11})

df = pd.read_csv('../data/processed/oecd_internet_gdp.csv')
print(f'Shape: {df.shape}')
df.head(10)

---
## 1. Basic Overview

In [ ]:
print('=== Dataset Info ===')
print(f'Countries : {df["country"].nunique()}')
print(f'Years     : {df["year"].min()} – {df["year"].max()}')
print(f'Total rows: {len(df)}')
print()
print('=== Descriptive Statistics ===')
df[['internet_usage_pct', 'gdp_per_capita_usd']].describe().round(2)

In [ ]:
print('=== Missing Values ===')
print(df.isnull().sum())

---
## 2. Internet Usage Over Time

In [ ]:
yearly_avg = df.groupby('year')[['internet_usage_pct', 'gdp_per_capita_usd']].mean()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(yearly_avg.index, yearly_avg['internet_usage_pct'], 'o-', color='steelblue', linewidth=2)
axes[0].set_title('Average Internet Usage (OECD, 2005–2020)')
axes[0].set_xlabel('Year')
axes[0].set_ylabel('% of individuals using internet')
axes[0].set_ylim(0, 100)

axes[1].plot(yearly_avg.index, yearly_avg['gdp_per_capita_usd'] / 1000, 'o-', color='coral', linewidth=2)
axes[1].set_title('Average GDP per Capita (OECD, 2005–2020)')
axes[1].set_xlabel('Year')
axes[1].set_ylabel('GDP per Capita (thousand USD)')

plt.tight_layout()
plt.savefig('../figures/01_trends_over_time.png', bbox_inches='tight')
plt.show()
print('Saved: figures/01_trends_over_time.png')

---
## 3. Country-Level Snapshot (2020)

In [ ]:
df_2020 = df[df['year'] == 2020].copy()

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# --- Left: Linear scale ---
scatter = axes[0].scatter(
    df_2020['internet_usage_pct'],
    df_2020['gdp_per_capita_usd'] / 1000,
    c=df_2020['gdp_per_capita_usd'], cmap='viridis', s=80, alpha=0.85, zorder=3
)
# OLS regression line
slope, intercept, r_val, p_val, se = stats.linregress(
    df_2020['internet_usage_pct'], df_2020['gdp_per_capita_usd'] / 1000
)
x_line = np.linspace(df_2020['internet_usage_pct'].min(), df_2020['internet_usage_pct'].max(), 100)
axes[0].plot(x_line, slope * x_line + intercept, 'r--', lw=2,
             label=f'OLS fit  R²={r_val**2:.3f}')
for _, row in df_2020.iterrows():
    if row['country'] in ['United States','Turkey','Luxembourg','Mexico','Norway','Germany','South Korea']:
        axes[0].annotate(row['country'],
                         (row['internet_usage_pct'], row['gdp_per_capita_usd']/1000),
                         fontsize=8, xytext=(4,4), textcoords='offset points')
axes[0].set_xlabel('Internet Usage (%)')
axes[0].set_ylabel('GDP per Capita (thousand USD)')
axes[0].set_title('Internet Usage vs GDP per Capita (2020)')
axes[0].legend(fontsize=9)
plt.colorbar(scatter, ax=axes[0], label='GDP per Capita (USD)')

# --- Right: Log-linear scale ---
df_2020['log_gdp'] = np.log(df_2020['gdp_per_capita_usd'])
slope_log, intercept_log, r_log, p_log, _ = stats.linregress(
    df_2020['internet_usage_pct'], df_2020['log_gdp']
)
axes[1].scatter(df_2020['internet_usage_pct'], df_2020['log_gdp'],
                color='steelblue', s=80, alpha=0.85, zorder=3)
axes[1].plot(x_line, slope_log * x_line + intercept_log, 'r--', lw=2,
             label=f'OLS fit (log GDP)  R²={r_log**2:.3f}')
axes[1].set_xlabel('Internet Usage (%)')
axes[1].set_ylabel('log(GDP per Capita)')
axes[1].set_title('Internet Usage vs log(GDP) (2020) — Better Fit')
axes[1].legend(fontsize=9)

plt.tight_layout()
plt.savefig('../figures/02_scatter_regression_2020.png', bbox_inches='tight')
plt.show()
print(f'Linear model  : R² = {r_val**2:.4f}, slope = {slope:.2f}')
print(f'Log-linear    : R² = {r_log**2:.4f}, slope = {slope_log:.4f}')

---
## 4. Correlation Analysis

In [ ]:
r_all, p_all = stats.pearsonr(df['internet_usage_pct'], df['gdp_per_capita_usd'])
print(f'Pearson r (all years): {r_all:.3f}  (p = {p_all:.4f})  R² = {r_all**2:.3f}')

yearly_corr = df.groupby('year').apply(
    lambda g: stats.pearsonr(g['internet_usage_pct'], g['gdp_per_capita_usd'])[0]
)

fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(yearly_corr.index, yearly_corr.values, color='steelblue', alpha=0.8)
ax.axhline(0, color='black', linewidth=0.8)
ax.set_xlabel('Year')
ax.set_ylabel('Pearson r')
ax.set_title('Yearly Pearson Correlation: Internet Usage vs GDP per Capita')
ax.set_ylim(-1, 1)
plt.tight_layout()
plt.savefig('../figures/03_yearly_correlation.png', bbox_inches='tight')
plt.show()

In [ ]:
df_log = df.copy()
df_log['log_gdp'] = np.log(df['gdp_per_capita_usd'])

corr_matrix = df_log[['internet_usage_pct', 'gdp_per_capita_usd', 'log_gdp', 'year']].corr()

fig, ax = plt.subplots(figsize=(7, 5))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', center=0,
            ax=ax, linewidths=0.5)
ax.set_title('Correlation Matrix')
plt.tight_layout()
plt.savefig('../figures/04_correlation_matrix.png', bbox_inches='tight')
plt.show()

---
## 5. Distribution Analysis

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 9))

axes[0, 0].hist(df['internet_usage_pct'], bins=25, color='steelblue', edgecolor='white', alpha=0.8)
axes[0, 0].set_xlabel('Internet Usage (%)')
axes[0, 0].set_title('Distribution of Internet Usage')

axes[0, 1].hist(df['gdp_per_capita_usd'] / 1000, bins=25, color='coral', edgecolor='white', alpha=0.8)
axes[0, 1].set_xlabel('GDP per Capita (thousand USD)')
axes[0, 1].set_title('Distribution of GDP per Capita')

axes[1, 0].hist(np.log(df['gdp_per_capita_usd']), bins=25, color='mediumseagreen', edgecolor='white', alpha=0.8)
axes[1, 0].set_xlabel('Log(GDP per Capita)')
axes[1, 0].set_title('Distribution of Log GDP per Capita (more symmetric)')

years_sample = [2005, 2008, 2011, 2014, 2017, 2020]
data_by_year = [df[df['year'] == y]['internet_usage_pct'].values for y in years_sample]
axes[1, 1].boxplot(data_by_year, labels=years_sample, patch_artist=True,
                   boxprops=dict(facecolor='steelblue', alpha=0.6))
axes[1, 1].set_xlabel('Year')
axes[1, 1].set_ylabel('Internet Usage (%)')
axes[1, 1].set_title('Internet Usage Distribution by Year')

plt.tight_layout()
plt.savefig('../figures/05_distributions.png', bbox_inches='tight')
plt.show()

In [ ]:
# Violin plots — richer distribution view than boxplots
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Internet usage by year (violin)
years_sample = [2005, 2008, 2011, 2014, 2017, 2020]
df_violin = df[df['year'].isin(years_sample)].copy()
df_violin['year'] = df_violin['year'].astype(str)

sns.violinplot(data=df_violin, x='year', y='internet_usage_pct',
               palette='Blues', inner='box', ax=axes[0])
axes[0].set_xlabel('Year')
axes[0].set_ylabel('Internet Usage (%)')
axes[0].set_title('Internet Usage Distribution by Year (Violin)')

# GDP per capita by internet group (violin)
df['internet_group'] = df['internet_usage_pct'].apply(
    lambda x: 'High (≥80%)' if x >= 80 else 'Low (<80%)'
)
sns.violinplot(data=df, x='internet_group', y='gdp_per_capita_usd',
               palette={'High (≥80%)': 'steelblue', 'Low (<80%)': 'coral'},
               inner='box', ax=axes[1])
axes[1].set_xlabel('Internet Usage Group')
axes[1].set_ylabel('GDP per Capita (USD)')
axes[1].set_title('GDP Distribution by Internet Usage Group')

plt.tight_layout()
plt.savefig('../figures/05b_violin_plots.png', bbox_inches='tight')
plt.show()
print('Saved: figures/05b_violin_plots.png')

---
## 6. Country Trajectories (Selected Countries)

In [ ]:
highlight = ['United States', 'Germany', 'Turkey', 'Mexico', 'South Korea', 'Norway']
df_hi = df[df['country'].isin(highlight)]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for country, grp in df_hi.groupby('country'):
    grp = grp.sort_values('year')
    axes[0].plot(grp['year'], grp['internet_usage_pct'], marker='o', markersize=3, label=country)
    axes[1].plot(grp['year'], grp['gdp_per_capita_usd'] / 1000, marker='o', markersize=3, label=country)

axes[0].set_title('Internet Usage Over Time (Selected Countries)')
axes[0].set_ylabel('Internet Usage (%)')
axes[0].legend(fontsize=8)

axes[1].set_title('GDP per Capita Over Time (Selected Countries)')
axes[1].set_ylabel('GDP per Capita (thousand USD)')
axes[1].legend(fontsize=8)

for ax in axes:
    ax.set_xlabel('Year')

plt.tight_layout()
plt.savefig('../figures/06_country_trajectories.png', bbox_inches='tight')
plt.show()

---
## 6.5. Country Rankings (2020)

In [ ]:
df_2020_rank = df[df['year'] == 2020].copy().sort_values('internet_usage_pct', ascending=True)

fig, axes = plt.subplots(1, 2, figsize=(16, 10))

# Internet usage ranking
colors_internet = ['#d73027' if v < 80 else '#4575b4' for v in df_2020_rank['internet_usage_pct']]
axes[0].barh(df_2020_rank['country'], df_2020_rank['internet_usage_pct'],
             color=colors_internet, alpha=0.85)
axes[0].axvline(80, color='red', linestyle='--', linewidth=1.5, label='80% threshold (H2)')
axes[0].set_xlabel('Internet Usage (%)')
axes[0].set_title('Internet Usage by Country (2020)')
axes[0].legend(fontsize=9)
for i, (val, country) in enumerate(zip(df_2020_rank['internet_usage_pct'], df_2020_rank['country'])):
    axes[0].text(val + 0.3, i, f'{val:.0f}%', va='center', fontsize=7.5)

# GDP ranking
df_2020_gdp = df[df['year'] == 2020].copy().sort_values('gdp_per_capita_usd', ascending=True)
axes[1].barh(df_2020_gdp['country'], df_2020_gdp['gdp_per_capita_usd'] / 1000,
             color='coral', alpha=0.85)
axes[1].set_xlabel('GDP per Capita (thousand USD)')
axes[1].set_title('GDP per Capita by Country (2020)')
for i, (val, country) in enumerate(zip(df_2020_gdp['gdp_per_capita_usd']/1000, df_2020_gdp['country'])):
    axes[1].text(val + 0.3, i, f'{val:.0f}k', va='center', fontsize=7.5)

plt.tight_layout()
plt.savefig('../figures/06b_country_rankings.png', bbox_inches='tight')
plt.show()
print('Saved: figures/06b_country_rankings.png')

---
## 7. Hypothesis Testing

Before running t-tests, we verify the **normality assumption** using the Shapiro-Wilk test and Q-Q plots.

### Assumption Checks: Normality

In [ ]:
high_group = df[df['internet_usage_pct'] >= 80]['gdp_per_capita_usd']
low_group  = df[df['internet_usage_pct'] <  80]['gdp_per_capita_usd']
x2005 = df[df['year'] == 2005].set_index('country_code')['internet_usage_pct']
x2020 = df[df['year'] == 2020].set_index('country_code')['internet_usage_pct']
common_idx = x2005.index.intersection(x2020.index)
diff_h3 = x2020.loc[common_idx] - x2005.loc[common_idx]

fig, axes = plt.subplots(2, 3, figsize=(15, 9))

samples = [
    (high_group,  'H2: High-Internet GDP', axes[0,0], axes[1,0]),
    (low_group,   'H2: Low-Internet GDP',  axes[0,1], axes[1,1]),
    (diff_h3,     'H3: 2020-2005 Diff',    axes[0,2], axes[1,2]),
]

for data, label, ax_hist, ax_qq in samples:
    # Histogram
    ax_hist.hist(data, bins=20, color='steelblue', edgecolor='white', alpha=0.8)
    stat, p = stats.shapiro(data)
    ax_hist.set_title(f'{label}\nShapiro-Wilk: W={stat:.3f}, p={p:.4f}')
    ax_hist.set_xlabel('Value')
    # Q-Q plot
    stats.probplot(data, dist='norm', plot=ax_qq)
    ax_qq.set_title(f'Q-Q Plot: {label}')

plt.tight_layout()
plt.savefig('../figures/07_normality_checks.png', bbox_inches='tight')
plt.show()
print('Note: Shapiro-Wilk p > 0.05 → normality not rejected → t-test assumptions hold.')

---
### H1: There is a significant positive correlation between internet usage and GDP per capita
- **H₀:** ρ = 0 (no correlation)
- **H₁:** ρ > 0 (positive correlation)
- **Tests:** Pearson (parametric) + Spearman (non-parametric)

In [ ]:
# Pearson
r, p = stats.pearsonr(df['internet_usage_pct'], df['gdp_per_capita_usd'])
r2 = r ** 2
print('=== H1: Pearson Correlation Test ===')
print(f'r  = {r:.4f}')
print(f'R² = {r2:.4f}  (internet usage explains {r2*100:.1f}% of variance in GDP)')
print(f'p  = {p:.6f}')
print(f'Result: {"Reject H₀ → significant positive correlation" if p < 0.05 and r > 0 else "Fail to reject H₀"}')
print()

# Spearman
rho, p_sp = stats.spearmanr(df['internet_usage_pct'], df['gdp_per_capita_usd'])
print('=== H1: Spearman Correlation Test ===')
print(f'ρ  = {rho:.4f}')
print(f'p  = {p_sp:.6f}')
print(f'Result: {"Reject H₀ → significant positive correlation" if p_sp < 0.05 and rho > 0 else "Fail to reject H₀"}')

In [ ]:
# Bootstrap 95% Confidence Interval for Pearson r
np.random.seed(42)
n_boot = 5000
boot_r = []
n = len(df)
for _ in range(n_boot):
    idx = np.random.choice(n, n, replace=True)
    r_b, _ = stats.pearsonr(df['internet_usage_pct'].iloc[idx],
                             df['gdp_per_capita_usd'].iloc[idx])
    boot_r.append(r_b)

ci_low, ci_high = np.percentile(boot_r, [2.5, 97.5])
print(f'Bootstrap 95% CI for Pearson r: [{ci_low:.4f}, {ci_high:.4f}]')
print(f'Observed r = {r:.4f}')

fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(boot_r, bins=60, color='steelblue', edgecolor='white', alpha=0.8)
ax.axvline(r, color='red', lw=2, label=f'Observed r = {r:.3f}')
ax.axvline(ci_low,  color='orange', lw=1.5, linestyle='--', label=f'95% CI [{ci_low:.3f}, {ci_high:.3f}]')
ax.axvline(ci_high, color='orange', lw=1.5, linestyle='--')
ax.set_xlabel('Bootstrap Pearson r')
ax.set_title('Bootstrap Distribution of Pearson r (n=5000)')
ax.legend()
plt.tight_layout()
plt.savefig('../figures/08_bootstrap_ci.png', bbox_inches='tight')
plt.show()

---
### H2: Countries with internet usage ≥80% have significantly higher GDP than those below 80%
- **H₀:** μ(high) = μ(low)
- **H₁:** μ(high) > μ(low)
- **Test:** Independent samples t-test + Cohen's d (effect size)

In [ ]:
threshold = 80
high = df[df['internet_usage_pct'] >= threshold]['gdp_per_capita_usd']
low  = df[df['internet_usage_pct'] <  threshold]['gdp_per_capita_usd']

t_stat, p_val = stats.ttest_ind(high, low, alternative='greater')

# Cohen's d
pooled_std = np.sqrt((high.std()**2 + low.std()**2) / 2)
cohens_d   = (high.mean() - low.mean()) / pooled_std

print('=== H2: Independent Samples t-test ===')
print(f'Threshold: {threshold}% internet usage')
print(f'High group (n={len(high):>4}): mean GDP = ${high.mean():>10,.0f}  std = ${high.std():>10,.0f}')
print(f'Low  group (n={len(low):>4}): mean GDP = ${low.mean():>10,.0f}  std = ${low.std():>10,.0f}')
print(f't-statistic = {t_stat:.4f}')
print(f'p-value     = {p_val:.6f}')
print(f"Cohen's d   = {cohens_d:.4f}  (|d|>0.8 → large effect)")
print(f'Result: {"Reject H₀ → high-internet countries have significantly higher GDP" if p_val < 0.05 else "Fail to reject H₀"}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 6))

df_h2 = df.copy()
df_h2['group'] = df_h2['internet_usage_pct'].apply(
    lambda x: 'High (≥80%)' if x >= 80 else 'Low (<80%)')

# Violin plot
sns.violinplot(data=df_h2, x='group', y='gdp_per_capita_usd',
               palette={'High (≥80%)': 'steelblue', 'Low (<80%)': 'coral'},
               inner='box', ax=axes[0])
sns.stripplot(data=df_h2, x='group', y='gdp_per_capita_usd',
              color='black', alpha=0.15, size=2.5, jitter=True, ax=axes[0])
axes[0].set_title(f"H2: GDP by Internet Group\nCohen's d = {cohens_d:.3f} (large effect)")
axes[0].set_xlabel('Internet Usage Group')
axes[0].set_ylabel('GDP per Capita (USD)')

# Means with 95% CI bars
groups  = ['Low (<80%)', 'High (≥80%)']
means   = [low.mean(), high.mean()]
ci95    = [1.96 * g.sem() for g in [low, high]]
colors  = ['coral', 'steelblue']
axes[1].bar(groups, [m/1000 for m in means], yerr=[e/1000 for e in ci95],
            color=colors, alpha=0.8, capsize=6, width=0.5)
axes[1].set_ylabel('Mean GDP per Capita (thousand USD)')
axes[1].set_title('Mean GDP per Group with 95% CI')
for i, (g, m) in enumerate(zip(groups, means)):
    axes[1].text(i, m/1000 + ci95[i]/1000 + 0.5, f'${m/1000:.1f}k', ha='center', fontsize=10)

plt.tight_layout()
plt.savefig('../figures/09_h2_violin_ci.png', bbox_inches='tight')
plt.show()

---
### H3: Internet usage has increased significantly from 2005 to 2020 across OECD countries
- **H₀:** μ(2005) = μ(2020)
- **H₁:** μ(2020) > μ(2005)
- **Test:** Paired samples t-test + Cohen's d

In [ ]:
df_2005 = df[df['year'] == 2005].set_index('country_code')['internet_usage_pct']
df_2020v = df[df['year'] == 2020].set_index('country_code')['internet_usage_pct']
common = df_2005.index.intersection(df_2020v.index)
x2005 = df_2005.loc[common]
x2020v = df_2020v.loc[common]

t_stat, p_val = stats.ttest_rel(x2020v, x2005, alternative='greater')

# Cohen's d for paired
diff = x2020v - x2005
cohens_d_h3 = diff.mean() / diff.std()

print('=== H3: Paired t-test (2005 vs 2020) ===')
print(f'Mean internet usage 2005: {x2005.mean():.1f}%')
print(f'Mean internet usage 2020: {x2020v.mean():.1f}%')
print(f'Mean increase: +{diff.mean():.1f} percentage points')
print(f't-statistic = {t_stat:.4f}')
print(f'p-value     = {p_val:.8f}')
print(f"Cohen's d   = {cohens_d_h3:.4f}  (|d|>0.8 → large effect)")
print(f'Result: {"Reject H₀ → internet usage has significantly increased" if p_val < 0.05 else "Fail to reject H₀"}')

In [ ]:
# Per-country internet usage change: 2005 → 2020
country_map = df[df['year'].isin([2005, 2020])].pivot_table(
    index='country', columns='year', values='internet_usage_pct'
).dropna()
country_map.columns = ['y2005', 'y2020']
country_map['delta'] = country_map['y2020'] - country_map['y2005']
country_map = country_map.sort_values('delta', ascending=True)

fig, axes = plt.subplots(1, 2, figsize=(16, 9))

# Delta bar chart
bar_colors = ['#4575b4' if d >= country_map['delta'].median() else '#d73027'
              for d in country_map['delta']]
axes[0].barh(country_map.index, country_map['delta'], color=bar_colors, alpha=0.85)
axes[0].axvline(country_map['delta'].mean(), color='black', linestyle='--', lw=1.5,
                label=f'Mean Δ = +{country_map["delta"].mean():.1f} pp')
axes[0].set_xlabel('Percentage Point Change (2005 → 2020)')
axes[0].set_title('Internet Usage Increase per Country (H3)')
axes[0].legend(fontsize=9)

# Paired scatter (slope chart)
for country in country_map.index:
    v05 = country_map.loc[country, 'y2005']
    v20 = country_map.loc[country, 'y2020']
    axes[1].plot([2005, 2020], [v05, v20], 'o-', color='steelblue', alpha=0.4, linewidth=1)
axes[1].plot([2005, 2020],
             [country_map['y2005'].mean(), country_map['y2020'].mean()],
             'o-', color='red', linewidth=3, label='OECD Average', zorder=5)
axes[1].set_xticks([2005, 2020])
axes[1].set_xlabel('Year')
axes[1].set_ylabel('Internet Usage (%)')
axes[1].set_title('Paired Internet Usage: 2005 vs 2020 (all countries)')
axes[1].legend(fontsize=9)

plt.tight_layout()
plt.savefig('../figures/10_h3_delta.png', bbox_inches='tight')
plt.show()

---
## 8. Regression Modelling

OLS regression to quantify the relationship and check residual assumptions.

In [ ]:
# OLS: GDP ~ Internet Usage (full panel)
slope_ols, intercept_ols, r_ols, p_ols, se_ols = stats.linregress(
    df['internet_usage_pct'], df['gdp_per_capita_usd']
)
df['gdp_pred']     = intercept_ols + slope_ols * df['internet_usage_pct']
df['residuals']    = df['gdp_per_capita_usd'] - df['gdp_pred']

# OLS: log(GDP) ~ Internet Usage
df['log_gdp'] = np.log(df['gdp_per_capita_usd'])
slope_log2, intercept_log2, r_log2, p_log2, se_log2 = stats.linregress(
    df['internet_usage_pct'], df['log_gdp']
)
df['log_gdp_pred']  = intercept_log2 + slope_log2 * df['internet_usage_pct']
df['log_residuals'] = df['log_gdp'] - df['log_gdp_pred']

print('=== OLS Regression: GDP ~ Internet Usage ===')
print(f'Intercept : {intercept_ols:,.1f}')
print(f'Slope     : {slope_ols:,.1f}  (each +1% internet → +${slope_ols:,.0f} GDP)')
print(f'R²        : {r_ols**2:.4f}')
print(f'p-value   : {p_ols:.6f}')
print()
print('=== OLS Regression: log(GDP) ~ Internet Usage ===')
print(f'Slope     : {slope_log2:.4f}  (each +1% internet → GDP grows by {(np.exp(slope_log2)-1)*100:.2f}%)')
print(f'R²        : {r_log2**2:.4f}  ← better fit (log model)')
print(f'p-value   : {p_log2:.6f}')

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 11))

x_plot = np.linspace(df['internet_usage_pct'].min(), df['internet_usage_pct'].max(), 200)

# Top-left: Linear OLS
axes[0,0].scatter(df['internet_usage_pct'], df['gdp_per_capita_usd']/1000,
                  alpha=0.2, s=15, color='steelblue')
axes[0,0].plot(x_plot, (intercept_ols + slope_ols*x_plot)/1000, 'r-', lw=2,
               label=f'OLS  R²={r_ols**2:.3f}')
axes[0,0].set_xlabel('Internet Usage (%)')
axes[0,0].set_ylabel('GDP per Capita (thousand USD)')
axes[0,0].set_title('OLS: GDP ~ Internet Usage (all years)')
axes[0,0].legend()

# Top-right: Log-linear OLS
axes[0,1].scatter(df['internet_usage_pct'], df['log_gdp'],
                  alpha=0.2, s=15, color='mediumseagreen')
axes[0,1].plot(x_plot, intercept_log2 + slope_log2*x_plot, 'r-', lw=2,
               label=f'OLS  R²={r_log2**2:.3f}')
axes[0,1].set_xlabel('Internet Usage (%)')
axes[0,1].set_ylabel('log(GDP per Capita)')
axes[0,1].set_title('OLS: log(GDP) ~ Internet Usage — Better Fit')
axes[0,1].legend()

# Bottom-left: Residuals vs fitted (linear)
axes[1,0].scatter(df['gdp_pred']/1000, df['residuals']/1000,
                  alpha=0.2, s=15, color='steelblue')
axes[1,0].axhline(0, color='red', lw=1.5, linestyle='--')
axes[1,0].set_xlabel('Fitted Values (thousand USD)')
axes[1,0].set_ylabel('Residuals (thousand USD)')
axes[1,0].set_title('Residuals vs Fitted — Linear Model')

# Bottom-right: Residuals vs fitted (log)
axes[1,1].scatter(df['log_gdp_pred'], df['log_residuals'],
                  alpha=0.2, s=15, color='mediumseagreen')
axes[1,1].axhline(0, color='red', lw=1.5, linestyle='--')
axes[1,1].set_xlabel('Fitted Values (log scale)')
axes[1,1].set_ylabel('Residuals (log scale)')
axes[1,1].set_title('Residuals vs Fitted — Log Model (more homoscedastic)')

plt.tight_layout()
plt.savefig('../figures/11_regression_analysis.png', bbox_inches='tight')
plt.show()

---
## 9. Summary of Results

In [ ]:
r, p = stats.pearsonr(df['internet_usage_pct'], df['gdp_per_capita_usd'])
rho, p_sp = stats.spearmanr(df['internet_usage_pct'], df['gdp_per_capita_usd'])

high = df[df['internet_usage_pct'] >= 80]['gdp_per_capita_usd']
low  = df[df['internet_usage_pct'] <  80]['gdp_per_capita_usd']
t2, p2 = stats.ttest_ind(high, low, alternative='greater')
cohens_d_h2 = (high.mean() - low.mean()) / np.sqrt((high.std()**2 + low.std()**2) / 2)

df_a = df[df['year'] == 2005].set_index('country_code')['internet_usage_pct']
df_b = df[df['year'] == 2020].set_index('country_code')['internet_usage_pct']
comm = df_a.index.intersection(df_b.index)
diff3 = df_b.loc[comm] - df_a.loc[comm]
t3, p3 = stats.ttest_rel(df_b.loc[comm], df_a.loc[comm], alternative='greater')
cohens_d_h3 = diff3.mean() / diff3.std()

slope_s, _, r_s, p_s, _ = stats.linregress(df['internet_usage_pct'], df['gdp_per_capita_usd'])

print('=' * 75)
print('HYPOTHESIS TESTING SUMMARY')
print('=' * 75)
print(f'{"Hypothesis":<42} {"Test":<22} {"Stat":>8} {"p-value":>10} {"Effect":>10} {"Result"}')
print('-' * 75)
print(f'{"H1: Internet ↑ correlates with GDP ↑":<42} {"Pearson r":<22} {r:>8.3f} {p:>10.4f} {"R²="+str(round(r**2,3)):>10}  CONFIRMED')
print(f'{"":<42} {"Spearman ρ":<22} {rho:>8.3f} {p_sp:>10.4f} {"":>10}')
print(f'{"H2: ≥80% internet → higher GDP":<42} {"Indep. t-test":<22} {t2:>8.3f} {p2:>10.4f} {"d="+str(round(cohens_d_h2,2)):>10}  CONFIRMED')
print(f'{"H3: Internet usage ↑ 2005→2020":<42} {"Paired t-test":<22} {t3:>8.3f} {p3:>10.6f} {"d="+str(round(cohens_d_h3,2)):>10}  CONFIRMED')
print('=' * 75)
print()
print('OLS Regression (full panel):')
print(f'  GDP = {_:.0f} + {slope_s:.1f} × internet_pct   R² = {r_s**2:.3f}')
print(f'  Interpretation: each +1% internet penetration → +${slope_s:,.0f} GDP per capita')